## Result analysis (accuracy over iterations)

In [1]:
%pip install --quiet scipy

Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
from pathlib import Path
from scipy import stats

REPO_ROOT = Path(".").resolve()
if "Paper" in str(REPO_ROOT):
    REPO_ROOT = REPO_ROOT.parent
ACC_FILE = REPO_ROOT / "output" / "output_qwen2p5-7b-instruct-3-fixed-seed" / "acc.txt"
BLOCK_SIZE = 5
OUTLIER_THRESHOLD = 0.10

acc = np.loadtxt(ACC_FILE, dtype=np.float64)
n = len(acc)
iterations = np.arange(1, n + 1, dtype=float)
print(f"Loaded {n} iterations from {ACC_FILE.name}")

Loaded 257 iterations from acc.txt


In [3]:
# 1. Spearman correlation (iteration vs accuracy) with scipy
rho, p_spearman = stats.spearmanr(iterations, acc)
print(f"Spearman ρ(iteration, accuracy) = {rho:.4f}, p = {p_spearman:.4e}")

Spearman ρ(iteration, accuracy) = 0.5261, p = 1.0670e-19


In [4]:
# 2. Slope per 10 iterations (linear trend)
slope, intercept, r_value, p_reg, se_slope = stats.linregress(iterations, acc)
slope_per_10 = slope * 10
r_squared = r_value ** 2
print(f"Slope (accuracy per 10 it.) = {slope_per_10:.4f}")
print(f"R² = {r_squared:.4f}")

Slope (accuracy per 10 it.) = 0.0036
R² = 0.1554


In [5]:
# 3. First 5 vs last 5
first_5 = acc[:BLOCK_SIZE]
last_5 = acc[-BLOCK_SIZE:]
mean_first = np.mean(first_5)
mean_last = np.mean(last_5)
diff = mean_last - mean_first
print(f"First 5:  mean = {mean_first:.4f}, SD = {np.std(first_5, ddof=1):.4f}")
print(f"Last 5:   mean = {mean_last:.4f}, SD = {np.std(last_5, ddof=1):.4f}")
print(f"Improvement (last − first 5) = {diff:.4f}")

First 5:  mean = 0.4399, SD = 0.0615
Last 5:   mean = 0.6754, SD = 0.0313
Improvement (last − first 5) = 0.2354


In [6]:
# 4. Last 5: mean ± SD
last_5_mean = np.mean(acc[-BLOCK_SIZE:])
last_5_std = np.std(acc[-BLOCK_SIZE:], ddof=1)
print(f"Last 5: {last_5_mean:.4f} ± {last_5_std:.4f}")

Last 5: 0.6754 ± 0.0313


In [7]:
# 5. Best and final accuracy
best_acc = np.max(acc)
best_iter = int(np.argmax(acc)) + 1
final_acc = float(acc[-1])
print(f"Best accuracy:  {best_acc:.4f} (iteration {best_iter})")
print(f"Final accuracy: {final_acc:.4f}")

Best accuracy:  0.7126 (iteration 257)
Final accuracy: 0.7126


In [8]:
# 6. Outliers (acc < threshold)
outliers = acc[acc < OUTLIER_THRESHOLD]
n_outliers = len(outliers)
print(f"Outliers (acc < {OUTLIER_THRESHOLD}): n = {n_outliers}")
if n_outliers > 0:
    print(f"Values: {np.round(outliers, 4).tolist()}")

Outliers (acc < 0.1): n = 0


In [9]:
# 7. acc of the first one iteration and the last one iteration
first_acc = acc[0]
last_acc = acc[-1]
print(f"First iteration accuracy: {first_acc:.4f}")
print(f"Last iteration accuracy: {last_acc:.4f}")
print(f"Improvement (last - first) = {last_acc - first_acc:.4f}")

First iteration accuracy: 0.3798
Last iteration accuracy: 0.7126
Improvement (last - first) = 0.3328
